# Kata 30 — Criterios Explícitos para Reducir Falsos Positivos

> Spec: [`specs/030-explicit-criteria/spec.md`](../../specs/030-explicit-criteria/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import json

client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

"Be conservative" no produce resultados consistentes. Criterios
categóricos con ejemplos concretos por nivel de severidad sí. Un alto
false positive rate en una categoría destruye la confianza en TODAS.

## 2. Por qué importa

Si el reviewer flagga "potential security issue" en código seguro 1 de
cada 5 veces, los devs ignoran todos los flags de seguridad — incluyendo
los reales. La precisión es prerequisito de utilidad.

## 3. Caso correcto — criterios categóricos con ejemplos

In [ ]:
SYSTEM_EXPLICIT = """Eres reviewer de código. Reporta findings sólo si cumplen UNO de estos criterios EXACTOS:

CATEGORÍA security.hardcoded_secret
  - Reporta SI: hay un literal string que parece API key, password, token (formato `sk-...`, `ghp_...`, hex >32 chars).
  - NO reportes SI: el valor viene de os.environ o de un config loader.

CATEGORÍA bug.null_deref
  - Reporta SI: se accede `obj.attr` u `obj[key]` sin chequear None previamente, cuando el valor proviene de una función que documenta retornar Optional.
  - NO reportes SI: hay un `assert obj is not None` o `if obj:` previo.

CATEGORÍA bug.unhandled_exception
  - Reporta SI: una operación I/O (file, network, db) está en un try sin except específico, o sin try.
  - NO reportes SI: la función documenta lanzar la excepción al caller.

NO reportes:
- Estilo (espacios, naming, line length).
- "Could be problematic" o "potentially". Sólo certezas.

Severidad: error (rompe runtime garantizado), warning (degrada en edge case). Sólo reporta error y warning.

Devuelve JSON: {"findings": [{"category": "...", "severity": "...", "line": N, "explanation": "..."}]}"""

SYSTEM_VAGUE = """Eres reviewer. Sé conservador. Reporta sólo findings de alta confianza.
Devuelve JSON: {"findings": [...]}"""

CODE = """def process_payment(user_id, amount):
    user = db.find_user(user_id)
    user.balance -= amount  # user puede ser None si find_user no encontró
    api_key = "sk-abc123def456ghi789"  # hardcoded
    response = requests.get(api_url)  # sin try
    return response.json()
"""

def review(system_prompt):
    resp = client.messages.create(
        system=system_prompt,
        messages=[{"role":"user","content": f"Revisa:\n```python\n{CODE}\n```\nSólo el JSON, sin texto adicional."}],
    )
    txt = next((b.text for b in resp.content if b.type == "text"), "")
    # extraer JSON del texto
    start = txt.find("{")
    end = txt.rfind("}")
    try:
        return json.loads(txt[start:end+1]) if start >= 0 and end > start else {"raw": txt}
    except json.JSONDecodeError:
        return {"raw": txt}

print("=== explícito ===")
print(json.dumps(review(SYSTEM_EXPLICIT), indent=2, ensure_ascii=False))

## 4. Anti-patrón — criterios vagos

In [ ]:
print("=== vago (be conservative) ===")
print(json.dumps(review(SYSTEM_VAGUE), indent=2, ensure_ascii=False))

Compara: el explícito reporta los 3 issues con categorías exactas. El
vago tiende a (a) listar issues de estilo no pedidos, (b) usar lenguaje
hedge ("could be", "potentially"), o (c) fallar al reportar el null
deref por "ser conservador".

### 4.1 Estrategia: deshabilitar categoría problemática

In [ ]:
# En producción, una categoría con 35% FP rate destruye confianza en las demás.
# Estrategia: deshabilitar la categoría problemática mientras se afina.

SYSTEM_DISABLED_HARDCODED = SYSTEM_EXPLICIT.replace(
    """CATEGORÍA security.hardcoded_secret
  - Reporta SI: hay un literal string que parece API key, password, token (formato `sk-...`, `ghp_...`, hex >32 chars).
  - NO reportes SI: el valor viene de os.environ o de un config loader.""",
    """CATEGORÍA security.hardcoded_secret — TEMPORALMENTE DESHABILITADA
  - NO reportes nada de esta categoría. La estamos calibrando."""
)

print("=== explícito sin categoría hardcoded_secret ===")
result = review(SYSTEM_DISABLED_HARDCODED)
print(json.dumps(result, indent=2, ensure_ascii=False))
# El finding del api_key debería desaparecer; los otros dos siguen.

## 5. Argumento de certificación

- **Vago vs explícito**: "be conservative" produce inconsistencia.
  Categorías con criterios "report SI / NO reportes SI" + ejemplos
  produce consistencia.
- **FP en una categoría destruye confianza en todas** — deshabilitar
  temporalmente la categoría problemática mientras se afina.
- **Severidad con ejemplos concretos** > "use your judgment".
- Conexión: complementa Kata 13 (CI review schema) y Kata 14 (few-shot
  como vehículo de criterios).

## 6. Auto-evaluación

**1. "Be conservative" — ¿por qué falla y qué lo reemplaza?**

Falla porque el modelo interpreta "conservative" distinto cada vez.
Reemplaza con criterios categóricos: "reporta SI X, NO reportes SI Y",
con ejemplos concretos de cada caso.

**2. Mi reviewer reporta security findings con 35 % FP rate y bug
findings con 5 %. ¿Qué hago primero?**

Deshabilitar security temporalmente. La pérdida de 35 % FP en security
está envenenando la utilidad del 95 % de bug findings correctos. Calibra
security offline; mantén bug.

**3. ¿Por qué severity-with-examples supera a "use your judgment"?**

"Judgment" no es enseñable; ejemplos sí. Mostrar 2 ejemplos de "esto es
error" + 2 de "esto no es error" calibra la distribución del modelo
hacia el operacional del equipo, no su default genérico.